In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is efined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q fastapi uvicorn pyngrok transformers==4.52.4 accelerate youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.2 MB/s eta 0:00:0000:01


In [3]:
import os
import socket
import threading
import time

from urllib.parse import urlparse, parse_qs

from fastapi import FastAPI, Request, HTTPException
import uvicorn

from pyngrok import ngrok, conf

from youtube_transcript_api import YouTubeTranscriptApi

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [4]:
def extract_video_id(url : str) -> str:
    parsed = urlparse(url)

    host = (parsed.hostname or "").lower()

    if host.startswith("www."):
        host = host[4:]

    # youtu.be/VIDEO_ID
    if host == "youtu.be":
        video_id = parsed.path.lstrip("/").split("/")[0]

        if video_id:
            return video_id

    # youtube.com/watch?v=VIDEO_ID
    if host in (
        "youtube.com",
        "m.youtube.com",
        "music.youtube.com"
    ):
        qs = parse_qs(parsed.query)

        if qs.get("v"):
            return qs["v"][0]

        # shorts / embed / live
        parts = parsed.path.strip("/").split("/")

        if len(parts) >= 2 and parts[0] in (
            "shorts",
            "embed",
            "live",
            "v"
        ):
            return parts[1]

    raise ValueError("Could not extract YouTube video ID")

In [5]:
def get_transcript(url : str) -> str:

    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()

    try:
        fetched = api.fetch(
            video_id,
            languages=["ar", "en"]
        )

        text = "\n".join(
            snippet.text
            for snippet in fetched
        )

        return text

    except Exception as e:
        raise Exception(
            f"Could not get transcript: {str(e)}"
        )


In [6]:
!pip install -q transformers accelerate sentencepiece

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# Device
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# =========================
# Model
# =========================

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded!")

Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded!


In [9]:
def generate_response(
    prompt,
    max_new_tokens=500
):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Arabic video summarizer. "
                "Your job is to understand the transcript, "
                "extract the important ideas, remove repetition, "
                "and never invent information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.1,
            repetition_penalty=1.15
        )

    generated_tokens = output[
        0
    ][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [10]:
import re

def clean_transcript(text):

    # Remove music markers
    text = re.sub(
        r"\[موسيقى\]",
        " ",
        text
    )

    # Remove speaker arrows
    text = text.replace(">>", " ")

    # Remove excessive spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [12]:
def split_text(
    text,
    chunk_size=180
):

    words = text.split()

    chunks = []

    for i in range(
        0,
        len(words),
        chunk_size
    ):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        chunks.append(chunk)

    return chunks

def summarize_chunk(chunk):

    prompt = f"""
اقرأ الجزء التالي من Transcript لفيديو باللغة العربية.

استخرج أهم المعلومات والأفكار الموجودة فيه فقط.

القواعد:
- لا تكرر نفس الفكرة.
- احذف الكلام غير المهم.
- لا تضف أي معلومة غير موجودة في النص.
- حافظ على أسماء الأشخاص والشركات والمصطلحات المهمة.
- لا تحاول تلخيص الكلام كلمة بكلمة.
- ركز على المعنى والأفكار.
- اكتب بالعربية المصرية الواضحة.
- لو الجزء يحتوي على مثال مهم، احتفظ به.

اكتب 3 إلى 6 نقاط مختصرة.

Transcript:

{chunk}
"""

    return generate_response(
        prompt,
        max_new_tokens=350
    )

In [16]:
def create_final_summary(summaries):

    combined = "\n\n".join(
        f"PART {i+1}:\n{summary}"
        for i, summary in enumerate(summaries)
    )

    prompt = f"""
أنت تقوم بإنشاء الملخص النهائي لفيديو عربي.

هذه مجموعة من ملخصات أجزاء مختلفة من نفس الفيديو:

{combined}

أنشئ ملخصًا نهائيًا مترابطًا.

المطلوب:

1. الفكرة الرئيسية للفيديو
2. أهم الأفكار والنقاط
3. أهم الأمثلة أو التجارب المذكورة
4. أهم الخلاصات العملية

القواعد:
- ادمج الأفكار المتكررة في نقطة واحدة.
- لا تكرر نفس المعلومة بصيغ مختلفة.
- لا تضف أي معلومات غير موجودة في النص.
- لا تذكر أن المعلومات جاءت من أجزاء مختلفة.
- حافظ على أسماء الأشخاص والشركات والأرقام المهمة.
- لا تجعل الملخص مجرد إعادة صياغة للجمل.
- استخرج المعنى العام للفيديو.
- اكتب بالعربية المصرية الواضحة.
- اجعل النتيجة سهلة القراءة ومختصرة.

استخدم الشكل التالي:

الفكرة الرئيسية:
...

أهم النقاط:
- ...
- ...
- ...

أهم الأمثلة:
- ...
- ...

الخلاصة:
...
"""

    return generate_response(
        prompt,
        max_new_tokens=900
    )

In [17]:
def summarize_text(text):

    print("Cleaning transcript...")

    text = clean_transcript(text)

    print("Splitting transcript...")

    chunks = split_text(
        text,
        chunk_size=180
    )

    print(
        f"Total chunks: {len(chunks)}"
    )

    summaries = []

    for i, chunk in enumerate(chunks):

        print(
            f"Summarizing chunk "
            f"{i+1}/{len(chunks)}..."
        )

        summary = summarize_chunk(
            chunk
        )

        summaries.append(
            summary
        )

    print(
        "Creating final summary..."
    )

    final_summary = create_final_summary(
        summaries
    )

    return final_summary

In [18]:
app = FastAPI()

API_KEY = os.getenv(
    "3K8DUVdBzWCUsFqJLTd48RT1nkg_46CYJ5ruHimtjBimuCSFg",
    "secret123"
)

In [20]:
@app.get("/")
async def root():

    return {
        "status": "online",
        "service": "YouTube Arabic Summarizer"
    }

In [21]:
@app.post("/generate")
async def generate(req: Request):

    # Authentication
    auth = req.headers.get("authorization")

    if auth != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    # Read JSON
    try:
        data = await req.json()
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Invalid JSON body"
        )

    youtube_url = data.get("youtube_url")

    if not youtube_url:
        raise HTTPException(
            status_code=400,
            detail="youtube_url is required"
        )

    try:
        print("=" * 60)
        print("NEW SUMMARIZATION REQUEST")
        print("=" * 60)

        # Get video ID
        video_id = extract_video_id(youtube_url)

        print(f"Video ID: {video_id}")

        # Get transcript
        print("Getting transcript...")

        transcript = get_transcript(youtube_url)

        print(
            f"Transcript length: {len(transcript)} characters"
        )

        if not transcript.strip():
            raise Exception("Transcript is empty")

        # Generate summary
        print("Generating summary...")

        summary = summarize_text(transcript)

        if not summary or not summary.strip():
            raise Exception(
                "Summary generation returned empty result"
            )

        print("Summary generated successfully.")
        print("=" * 60)

        return {
            "success": True,
            "video_id": video_id,
            "summary": summary
        }

    except HTTPException:
        raise

    except Exception as e:
        print(f"ERROR: {str(e)}")

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


In [22]:
def free_port():

    s = socket.socket()

    s.bind(("", 0))

    port = s.getsockname()[1]

    s.close()

    return port

In [23]:
port = free_port()

NGROK_TOKEN = os.getenv(
    "NGROK_TOKEN"
)

conf.get_default().auth_token = "3K8DUVdBzWCUsFqJLTd48RT1nkg_46CYJ5ruHimtjBimuCSFg"

public_url = ngrok.connect(port).public_url

print("================================")
print("PUBLIC API URL:")
print(public_url)
print("================================")

PUBLIC API URL:
https://ecard-pregnancy-agenda.ngrok-free.dev


In [24]:
def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )


thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

time.sleep(2)

print(
    f"API running on {public_url}"
)

INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:37479 (Press CTRL+C to quit)


API running on https://ecard-pregnancy-agenda.ngrok-free.dev
NEW SUMMARIZATION REQUEST
Video ID: 0jBEcxTenvE
Getting transcript...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Transcript length: 1085 characters
Generating summary...
Cleaning transcript...
Splitting transcript...
Total chunks: 2
Summarizing chunk 1/2...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Transcript length: 1085 characters
Generating summary...
Cleaning transcript...
Splitting transcript...
Total chunks: 2
Summarizing chunk 1/2...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Summarizing chunk 2/2...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Creating final summary...
Summary generated successfully.
INFO:     41.239.109.253:0 - "POST /generate HTTP/1.1" 200 OK
